In [2]:
print("Hello Jupyter")

Hello Jupyter


In [3]:
import requests
import json

url = "https://api.scrydex.com/v1/pokemon/cards"
params = {
    "page": 1,
    "pageSize": 5
}

response = requests.get(url, params=params)
print(response.status_code)

data = response.json()
print(type(data))
print(data.keys() if isinstance(data, dict) else "Not a dict")

401
<class 'dict'>
dict_keys(['error'])


In [5]:
!pip install pandas scikit-learn

In [1]:
import json
import re
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
data_folder = Path("cards\en")   # or Path(r"C:\Users\YourName\pokemon_project\data")

all_cards = []

for file_path in data_folder.glob("*.json"):
    with open(file_path, "r", encoding="utf-8") as f:
        content = json.load(f)
        
        # Case 1: file itself is a list of cards
        if isinstance(content, list):
            all_cards.extend(content)
        
        # Case 2: file is a dict, and cards are under "data"
        elif isinstance(content, dict) and "data" in content:
            all_cards.extend(content["data"])

print("Total cards loaded:", len(all_cards))
print("Sample keys:", all_cards[0].keys() if all_cards else "No data")

Total cards loaded: 19465
Sample keys: dict_keys(['id', 'name', 'supertype', 'subtypes', 'level', 'hp', 'types', 'evolvesFrom', 'abilities', 'attacks', 'weaknesses', 'retreatCost', 'convertedRetreatCost', 'number', 'artist', 'rarity', 'flavorText', 'nationalPokedexNumbers', 'legalities', 'images'])


<>:1: SyntaxWarning: "\e" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\e"? A raw string is also an option.
<>:1: SyntaxWarning: "\e" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\e"? A raw string is also an option.
C:\Users\fandy\AppData\Local\Temp\ipykernel_108192\624502024.py:1: SyntaxWarning: "\e" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\e"? A raw string is also an option.
  data_folder = Path("cards\en")   # or Path(r"C:\Users\YourName\pokemon_project\data")


In [3]:
print(all_cards[0])

{'id': 'base1-1', 'name': 'Alakazam', 'supertype': 'Pokémon', 'subtypes': ['Stage 2'], 'level': '42', 'hp': '80', 'types': ['Psychic'], 'evolvesFrom': 'Kadabra', 'abilities': [{'name': 'Damage Swap', 'text': "As often as you like during your turn (before your attack), you may move 1 damage counter from 1 of your Pokémon to another as long as you don't Knock Out that Pokémon. This power can't be used if Alakazam is Asleep, Confused, or Paralyzed.", 'type': 'Pokémon Power'}], 'attacks': [{'name': 'Confuse Ray', 'cost': ['Psychic', 'Psychic', 'Psychic'], 'convertedEnergyCost': 3, 'damage': '30', 'text': 'Flip a coin. If heads, the Defending Pokémon is now Confused.'}], 'weaknesses': [{'type': 'Psychic', 'value': '×2'}], 'retreatCost': ['Colorless', 'Colorless', 'Colorless'], 'convertedRetreatCost': 3, 'number': '1', 'artist': 'Ken Sugimori', 'rarity': 'Rare Holo', 'flavorText': 'Its brain can outperform a supercomputer. Its intelligence quotient is said to be 5000.', 'nationalPokedexNumbe

In [4]:
def build_input_text(card):
    parts = []

    if card.get("name"):
        parts.append(f"Name: {card['name']}")

    if card.get("hp"):
        parts.append(f"HP: {card['hp']}")

    if card.get("types"):
        parts.append("Types: " + ", ".join(card["types"]))

    for ability in card.get("abilities", []):
        if ability.get("name"):
            parts.append(f"Ability Name: {ability['name']}")
        if ability.get("text"):
            parts.append(f"Ability Text: {ability['text']}")

    for attack in card.get("attacks", []):
        if attack.get("name"):
            parts.append(f"Attack Name: {attack['name']}")
        if attack.get("damage"):
            parts.append(f"Attack Damage: {attack['damage']}")
        if attack.get("text"):
            parts.append(f"Attack Text: {attack['text']}")

    for rule in card.get("rules", []):
        parts.append(f"Rule: {rule}")

    return "\n".join(parts).strip()

In [5]:
def build_output_json(card):
    return {
        "name": card.get("name"),
        "hp": card.get("hp"),
        "types": card.get("types", []),
        "abilities": [
            {
                "name": a.get("name"),
                "text": a.get("text")
            }
            for a in card.get("abilities", [])
        ],
        "attacks": [
            {
                "name": atk.get("name"),
                "damage": atk.get("damage"),
                "text": atk.get("text")
            }
            for atk in card.get("attacks", [])
        ],
        "weaknesses": [
            {
                "type": w.get("type"),
                "value": w.get("value")
            }
            for w in card.get("weaknesses", [])
        ],
        "retreatCost": card.get("retreatCost", [])
    }

In [6]:
def has_useful_content(card):
    return bool(
        card.get("abilities")
        or card.get("attacks")
        or card.get("weaknesses")
        or card.get("retreatCost")
    )

In [7]:
filtered_cards = [c for c in all_cards if c.get("supertype") == "Pokémon"]

In [8]:
dataset = []

for card in filtered_cards:
    if not has_useful_content(card):
        continue

    input_text = build_input_text(card)
    output_json = build_output_json(card)

    if not input_text:
        continue

    dataset.append({
        "input": input_text,
        "output": output_json
    })

In [9]:
# 1. load data → all_cards
# (you already did this)

# 2. deduplicate
unique_cards = {}
for card in all_cards:
    card_id = card.get("id")
    if card_id:
        unique_cards[card_id] = card

all_cards = list(unique_cards.values())

# 3. filter Pokémon cards (THIS WAS MISSING)
filtered_cards = [
    c for c in all_cards 
    if c.get("supertype") == "Pokémon"
]

print("Total Pokémon cards:", len(filtered_cards))

Total Pokémon cards: 16456


In [10]:
print(filtered_cards[0].keys())

dict_keys(['id', 'name', 'supertype', 'subtypes', 'level', 'hp', 'types', 'evolvesFrom', 'abilities', 'attacks', 'weaknesses', 'retreatCost', 'convertedRetreatCost', 'number', 'artist', 'rarity', 'flavorText', 'nationalPokedexNumbers', 'legalities', 'images'])


In [11]:
dataset = []

for card in filtered_cards:
    if not has_useful_content(card):
        continue

    input_text = build_input_text(card)
    output_json = build_output_json(card)

    if not input_text:
        continue

    dataset.append({
        "input": input_text,
        "output": output_json
    })

In [12]:
dataset = dataset[:16000]

In [13]:
print(dataset[0].keys())

dict_keys(['input', 'output'])


In [14]:
import random
random.seed(42)

def input_compact(sample):
    """
    Turn multi-line input into a more compact format.
    Keep labels, but merge related content into fewer lines.
    """
    out = sample["output"]

    parts = []

    # Basic info in one line
    basic = []
    if out.get("name"):
        basic.append(f"Name: {out['name']}")
    if out.get("hp"):
        basic.append(f"HP: {out['hp']}")
    if out.get("types"):
        basic.append(f"Types: {', '.join(out['types'])}")
    if basic:
        parts.append(" | ".join(basic))

    # Abilities
    for ab in out.get("abilities", []):
        ab_name = ab.get("name", "")
        ab_text = ab.get("text", "")
        parts.append(f"Ability: {ab_name} - {ab_text}")

    # Attacks
    for atk in out.get("attacks", []):
        atk_name = atk.get("name", "")
        atk_damage = atk.get("damage", "")
        atk_text = atk.get("text", "")
        parts.append(f"Attack: {atk_name} | Damage: {atk_damage} | Effect: {atk_text}")

    # Weaknesses
    if out.get("weaknesses"):
        weak_str = "; ".join(
            f"{w.get('type', '')} {w.get('value', '')}" for w in out["weaknesses"]
        )
        parts.append(f"Weakness: {weak_str}")

    # Retreat cost
    if out.get("retreatCost"):
        parts.append(f"Retreat: {', '.join(out['retreatCost'])}")

    return "\n".join(parts)


def input_minimal_shuffle(sample):
    """
    Slightly shuffle sections and use shorter labels.
    """
    out = sample["output"]

    sections = []

    # Shorter labels
    if out.get("name"):
        sections.append(f"{out['name']}")
    if out.get("hp"):
        sections.append(f"HP {out['hp']}")
    if out.get("types"):
        sections.append(f"Type {', '.join(out['types'])}")

    for ab in out.get("abilities", []):
        sections.append(f"Ability {ab.get('name', '')}: {ab.get('text', '')}")

    for atk in out.get("attacks", []):
        sections.append(
            f"Attack {atk.get('name', '')}, {atk.get('damage', '')}: {atk.get('text', '')}"
        )

    if out.get("weaknesses"):
        weak_str = "; ".join(
            f"{w.get('type', '')} {w.get('value', '')}" for w in out["weaknesses"]
        )
        sections.append(f"Weak {weak_str}")

    if out.get("retreatCost"):
        sections.append(f"Retreat {', '.join(out['retreatCost'])}")

    # Slight shuffle: keep first 3 basic fields near top, shuffle the rest
    head = sections[:3]
    tail = sections[3:]
    random.shuffle(tail)

    return "\n".join(head + tail)


def build_varied_input(sample):
    """
    Randomly choose one input style.
    """
    style = random.choice(["original", "compact", "minimal_shuffle"])

    if style == "original":
        return sample["input"]
    elif style == "compact":
        return input_compact(sample)
    else:
        return input_minimal_shuffle(sample)

In [15]:
import json

def make_unique_key(sample):
    out = sample["output"]

    name = out.get("name", "").strip().lower()

    abilities = out.get("abilities", [])
    abilities_key = json.dumps(
        abilities,
        sort_keys=True,
        ensure_ascii=False
    )

    return (name, abilities_key)


unique_dataset = []
seen = set()

for sample in dataset:
    key = make_unique_key(sample)

    if key in seen:
        continue

    seen.add(key)
    unique_dataset.append(sample)

print("Before dedup:", len(dataset))
print("After dedup:", len(unique_dataset))
print("Removed duplicates:", len(dataset) - len(unique_dataset))

dataset = unique_dataset

Before dedup: 16000
After dedup: 4792
Removed duplicates: 11208


In [16]:
from sklearn.model_selection import train_test_split
import copy

# Step 1: start from your original dataset before adding variations
original_dataset = copy.deepcopy(dataset)

# Step 2: split first
train_dataset, test_dataset = train_test_split(
    original_dataset,
    test_size=0.2,
    random_state=88
)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

Train size: 3833
Test size: 959


In [19]:
train_dataset_final = []

for sample in train_dataset:
    train_dataset_final.append({
        "input": build_varied_input(sample),
        "output": sample["output"]
    })

print("Train size:", len(train_dataset_final))

Train size: 3833


In [20]:
test_dataset_final = []

for sample in test_dataset:
    test_dataset_final.append({
        "input": build_varied_input(sample),
        "output": sample["output"]
    })

print("Test size:", len(test_dataset_final))

Test size: 959


In [22]:
print(train_dataset_final[10]['input'])

Name: Klefki | HP: 70 | Types: Fairy
Ability: Wonder Lock - Once during your turn (before your attack), if this Pokémon is on your Bench, you may discard all cards attached to this Pokémon and attach it to 1 of your Pokémon as a Pokémon Tool card. Prevent any damage done to the Pokémon this card is attached to by attacks from your opponent's Mega Evolution Pokémon. If this card is attached to a Pokémon, discard this card at the end of your opponent's turn.
Attack: Fairy Wind | Damage: 30 | Effect: 
Weakness: Metal ×2
Retreat: Colorless


In [23]:
print(train_dataset_final[10]['output'])

{'name': 'Klefki', 'hp': '70', 'types': ['Fairy'], 'abilities': [{'name': 'Wonder Lock', 'text': "Once during your turn (before your attack), if this Pokémon is on your Bench, you may discard all cards attached to this Pokémon and attach it to 1 of your Pokémon as a Pokémon Tool card. Prevent any damage done to the Pokémon this card is attached to by attacks from your opponent's Mega Evolution Pokémon. If this card is attached to a Pokémon, discard this card at the end of your opponent's turn."}], 'attacks': [{'name': 'Fairy Wind', 'damage': '30', 'text': ''}], 'weaknesses': [{'type': 'Metal', 'value': '×2'}], 'retreatCost': ['Colorless']}


In [24]:
import json

with open("train_dataset.jsonl", "w", encoding="utf-8") as f:
    for item in train_dataset_final:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

In [25]:
with open("test_dataset.jsonl", "w", encoding="utf-8") as f:
    for item in test_dataset_final:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

In [22]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import json
import re

In [25]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

In [26]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

C:\Users\fandy\pokemon-nlp-project\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\fandy\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [27]:
BASELINE_PROMPT = """
Extract the key information from the following Pokémon card text and return ONLY valid JSON.

Required keys:
name, hp, types, abilities, attacks, weaknesses, retreatCost

Rules:
- Return JSON only
- Do not include explanation
- abilities must be a list of objects with keys: name, text
- attacks must be a list of objects with keys: name, damage, text
- weaknesses must be a list of objects with keys: type, value
- retreatCost must be a list of strings

Card text:
{card_text}
"""

In [29]:
# Try 1 input to make sure the model is working
sample = test_dataset_final[0]

prompt = BASELINE_PROMPT.format(card_text=sample["input"])

messages = [
    {"role": "system", "content": "You are an information extraction assistant that returns JSON only."},
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=300,
    do_sample=False
)

generated_ids = [
    output_ids[len(input_ids):]
    for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

result = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(result)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


{
  "name": "Burmy",
  "hp": 40,
  "types": ["Grass"],
  "abilities": [
    {
      "name": "Cloak Evolution",
      "text": "Burmy Sandy Cloak can evolve during the turn you play it."
    }
  ],
  "attacks": [
    {
      "name": "Tackle",
      "damage": 20,
      "text": ""
    }
  ],
  "weaknesses": [
    {
      "type": "Fire",
      "value": "+10"
    }
  ],
  "retreatCost": []
}


In [32]:
# execute 100 inputs to model
import json

baseline_results = []

for sample in test_dataset_final[:100]:   # start with 20 first

    prompt = BASELINE_PROMPT.format(card_text=sample["input"])

    messages = [
        {"role": "system", "content": "You are an information extraction assistant that returns JSON only."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=300,
        do_sample=False
    )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    result = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    baseline_results.append({
        "input": sample["input"],
        "ground_truth": sample["output"],
        "prediction_raw": result
    })

with open("baseline_results_fetch.jsonl", "w", encoding="utf-8") as f:
        for item in baseline_results:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
print("Done:", len(baseline_results))

Done: 100


In [33]:
# calculate valid json rate and accuracy
import json
import re

def safe_parse_json(text):
    try:
        return json.loads(text)
    except:
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except:
                return None
        return None

def normalize(obj):
    if isinstance(obj, dict):
        return {k: normalize(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [normalize(x) for x in obj]
    if isinstance(obj, (int, float)):
        return str(obj)
    return obj

fields = ["name", "hp", "types", "abilities", "attacks", "weaknesses", "retreatCost"]

valid_json_count = 0
exact_match_count = 0
field_correct = {f: 0 for f in fields}

for r in baseline_results:
    pred = safe_parse_json(r["prediction_raw"])
    gt = r["ground_truth"]

    if pred is not None:
        valid_json_count += 1

        pred_norm = normalize(pred)
        gt_norm = normalize(gt)

        if pred_norm == gt_norm:
            exact_match_count += 1

        for f in fields:
            if pred_norm.get(f) == gt_norm.get(f):
                field_correct[f] += 1

total = len(baseline_results)

print("Total samples:", total)
print("Valid JSON Rate:", round(valid_json_count / total * 100, 2), "%")
print("Exact Match Accuracy:", round(exact_match_count / total * 100, 2), "%")

for f in fields:
    print(f"{f} Accuracy:", round(field_correct[f] / total * 100, 2), "%")

Total samples: 100
Valid JSON Rate: 87.0 %
Exact Match Accuracy: 0.0 %
name Accuracy: 83.0 %
hp Accuracy: 87.0 %
types Accuracy: 87.0 %
abilities Accuracy: 17.0 %
attacks Accuracy: 22.0 %
weaknesses Accuracy: 43.0 %
retreatCost Accuracy: 16.0 %
